# Multiple Cells - Updated

In [4]:
"""
GPT-2 Fine-tuning for Nepali Summarization - Modular Version
"""

import os
import json
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from tqdm import tqdm
import sentencepiece as spm
import math
import csv
import re
from rouge_score import rouge_scorer
from rouge_score.tokenizers import Tokenizer

# ============================================================================
# CONFIGURATION
# ============================================================================

@dataclass
class Config:
    # Paths
    CHECKPOINT_PATH: str = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\new-models\bpe16_updated_19k.pt"
    TOKENIZER_PATH: str = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\new-tokenizers\bpe-16-updated.model"
    TRAIN_JSONL: str = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\train_plus_val_norm.jsonl"
    TEST_JSONL: str = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\test_norm.jsonl"
    OUTPUT_DIR: str = "finetuned-summarization"
    
    # Model settings
    BLOCK_SIZE: int = 1024
    MAX_SOURCE_LENGTH: int = 768
    MAX_TARGET_LENGTH: int = 128  # Reduced from 256
    
    # Freezing
    FREEZE_EMBEDDINGS: bool = False
    FREEZE_BLOCKS: list = None
    
    # Training
    NUM_EPOCHS: int = 5
    BATCH_SIZE: int = 8
    GRADIENT_ACCUMULATION_STEPS: int = 4
    LEARNING_RATE: float = 1e-5
    WARMUP_RATIO: float = 0.1
    MAX_GRAD_NORM: float = 1.0
    WEIGHT_DECAY: float = 0.01
    
    # Generation
    GENERATION_TOP_K: int = 40
    GENERATION_TEMPERATURE: float = 0.7
    MAX_NEW_TOKENS: int = 64  # Reduced from 128
    
    # Evaluation
    EVAL_STEPS: int = 100
    ROUGE_SAMPLE_SIZE: int = 100
    MAX_CONSECUTIVE_NANS: int = 3
    
    def __post_init__(self):
        if self.FREEZE_BLOCKS is None:
            self.FREEZE_BLOCKS = list(range(8))
        os.makedirs(self.OUTPUT_DIR, exist_ok=True)

PROMPT_TEMPLATE = "यो लेखको संक्षेप गर्नुहोस्:\n{text}\nसारांश:\n"

# ============================================================================
# MODEL ARCHITECTURE
# ============================================================================

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.c_attn = nn.Linear(cfg.n_embd, 3 * cfg.n_embd)
        self.c_proj = nn.Linear(cfg.n_embd, cfg.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1
        self.n_head = cfg.n_head
        self.n_embd = cfg.n_embd

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.c_proj(y.transpose(1, 2).contiguous().view(B, T, C))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.c_fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 16384
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.config = cfg
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(cfg.vocab_size, cfg.n_embd),
            wpe=nn.Embedding(cfg.block_size, cfg.n_embd),
            h=nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)]),
            ln_f=nn.LayerNorm(cfg.n_embd),
        ))
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, input_ids, labels=None):
        B, T = input_ids.size()
        pos = torch.arange(0, T, dtype=torch.long, device=input_ids.device)
        x = self.transformer.wte(input_ids) + self.transformer.wpe(pos)
        
        for block in self.transformer.h:
            x = block(x)
        
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[..., :-1, :].contiguous().view(-1, logits.size(-1)),
                labels[..., 1:].contiguous().view(-1)
            )
        
        return loss, logits

In [5]:
# ============================================================================
# DATASET
# ============================================================================

class SummarizationDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, block_size):
        self.tokenizer = tokenizer
        self.block_size = block_size
        self.data = []
        
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    item = json.loads(line)
                    if 'text' in item and 'summary' in item:
                        self.data.append(item)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        prompt = PROMPT_TEMPLATE.format(text=item['text'])
        full_text = prompt + item['summary']
        
        tokens = self.tokenizer.encode(full_text)
        if len(tokens) > self.block_size:
            tokens = tokens[:self.block_size]
        
        input_ids = torch.tensor(tokens, dtype=torch.long)
        labels = input_ids.clone()
        prompt_len = len(self.tokenizer.encode(prompt))
        labels[:prompt_len] = -100
        
        return {'input_ids': input_ids, 'labels': labels}

def collate_fn(batch):
    max_len = max(len(item['input_ids']) for item in batch)
    input_ids = []
    labels = []
    
    for item in batch:
        pad_len = max_len - len(item['input_ids'])
        input_ids.append(torch.cat([item['input_ids'], torch.zeros(pad_len, dtype=torch.long)]))
        labels.append(torch.cat([item['labels'], torch.full((pad_len,), -100, dtype=torch.long)]))
    
    return {
        'input_ids': torch.stack(input_ids),
        'labels': torch.stack(labels)
    }

# ============================================================================
# NEPALI ROUGE
# ============================================================================

class NepaliTokenizer(Tokenizer):
    def tokenize(self, text):
        text = text.replace("।", "")
        text = re.sub(r"[^\w\s]", "", text)
        return text.split()

def get_rouge_scorer():
    return rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        tokenizer=NepaliTokenizer(),
        use_stemmer=False
    )

In [6]:
# ============================================================================
# GENERATION & EVALUATION
# ============================================================================

def generate_summary(model, tokenizer, text, cfg, device):
    """Generate summary using top-k sampling"""
    model.eval()
    prompt = PROMPT_TEMPLATE.format(text=text)
    tokens = tokenizer.encode(prompt)
    input_ids = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)
    generated = input_ids.clone()
    
    with torch.no_grad():
        for _ in range(cfg.MAX_NEW_TOKENS):
            if generated.size(1) >= model.config.block_size:
                break
            
            _, logits = model(generated)
            logits = logits[:, -1, :] / cfg.GENERATION_TEMPERATURE
            
            top_k_logits, top_k_indices = torch.topk(logits, min(cfg.GENERATION_TOP_K, logits.size(-1)))
            probs = F.softmax(top_k_logits, dim=-1)
            next_token = torch.gather(top_k_indices, -1, torch.multinomial(probs, 1))
            
            generated = torch.cat([generated, next_token], dim=1)
            if next_token.item() == 0:
                break
    
    model.train()
    return tokenizer.decode(generated[0, len(tokens):].tolist())

def evaluate_loss(model, dataloader, device):
    """Compute average loss"""
    model.eval()
    total_loss, count = 0.0, 0
    
    with torch.no_grad():
        for batch in dataloader:
            try:
                loss, _ = model(batch['input_ids'].to(device), batch['labels'].to(device))
                if not torch.isnan(loss) and not torch.isinf(loss):
                    total_loss += loss.item()
                    count += 1
            except RuntimeError:
                continue
    
    model.train()
    return total_loss / count if count > 0 else float('nan')

def compute_rouge(model, dataset, tokenizer, cfg, device):
    """Compute ROUGE scores"""
    scorer = get_rouge_scorer()
    scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    
    eval_size = min(cfg.ROUGE_SAMPLE_SIZE, len(dataset))
    
    for i in tqdm(range(eval_size), desc="ROUGE"):
        item = dataset.data[i]
        pred = generate_summary(model, tokenizer, item['text'], cfg, device)
        ref = item['summary']
        
        result = scorer.score(ref, pred)
        scores['rouge1'].append(result['rouge1'].fmeasure)
        scores['rouge2'].append(result['rouge2'].fmeasure)
        scores['rougeL'].append(result['rougeL'].fmeasure)
    
    return {k: sum(v) / len(v) * 100 for k, v in scores.items()}

# ============================================================================
# TRAINING UTILITIES
# ============================================================================

def setup_model(cfg, device):
    """Load and freeze model"""
    sp = spm.SentencePieceProcessor()
    sp.load(cfg.TOKENIZER_PATH)
    
    checkpoint = torch.load(cfg.CHECKPOINT_PATH, map_location=device, weights_only=False)
    model = GPT(checkpoint['config'])
    model.load_state_dict(checkpoint['model'])
    model.to(device)
    
    # Freeze position embeddings
    for param in model.transformer.wpe.parameters():
        param.requires_grad = False
    
    # Freeze specified blocks
    for block_idx in cfg.FREEZE_BLOCKS:
        for param in model.transformer.h[block_idx].parameters():
            param.requires_grad = False
    
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    
    print(f"✓ Model loaded: {checkpoint['step']} steps, val_loss={checkpoint['val_loss']:.4f}")
    print(f"✓ Trainable: {trainable:,}/{total:,} ({100*trainable/total:.1f}%)")
    print(f"✓ Frozen: Position embeddings + Blocks {cfg.FREEZE_BLOCKS[0]}-{cfg.FREEZE_BLOCKS[-1]}")
    
    return model, sp

def get_lr(step, cfg, total_steps, warmup_steps):
    """Cosine learning rate schedule with warmup"""
    if step < warmup_steps:
        return cfg.LEARNING_RATE * (step + 1) / warmup_steps
    if step >= total_steps:
        return cfg.LEARNING_RATE * 0.1
    decay_ratio = (step - warmup_steps) / (total_steps - warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return cfg.LEARNING_RATE * 0.1 + coeff * (cfg.LEARNING_RATE * 0.9)

def save_checkpoint(model, optimizer, epoch, step, train_loss, eval_loss, rouge_scores, cfg, name):
    """Save model checkpoint"""
    path = os.path.join(cfg.OUTPUT_DIR, name)
    torch.save({
        'model': model.state_dict(),
        'config': model.config,
        'optimizer': optimizer.state_dict(),
        'epoch': epoch,
        'global_step': step,
        'train_loss': train_loss,
        'eval_loss': eval_loss,
        'rouge_scores': rouge_scores,
    }, path)
    return path

In [ ]:
# ============================================================================
# MAIN TRAINING LOOP
# ============================================================================

def train():
    cfg = Config()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    
    # Setup
    model, sp = setup_model(cfg, device)
    train_dataset = SummarizationDataset(cfg.TRAIN_JSONL, sp, cfg.BLOCK_SIZE)
    test_dataset = SummarizationDataset(cfg.TEST_JSONL, sp, cfg.BLOCK_SIZE)
    
    train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    
    print(f"✓ Train: {len(train_dataset)} samples, {len(train_loader)} batches")
    print(f"✓ Test: {len(test_dataset)} samples")
    
    # Optimizer
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg.LEARNING_RATE,
        betas=(0.9, 0.999),
        weight_decay=cfg.WEIGHT_DECAY
    )
    
    total_steps = cfg.NUM_EPOCHS * len(train_loader) // cfg.GRADIENT_ACCUMULATION_STEPS
    warmup_steps = int(cfg.WARMUP_RATIO * total_steps)
    
    print(f"✓ Total steps: {total_steps}, Warmup: {warmup_steps}")
    
    # CSV logging
    csv_path = os.path.join(cfg.OUTPUT_DIR, "training_metrics.csv")
    csv_file = open(csv_path, 'w', newline='', encoding='utf-8')
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(['step', 'epoch', 'train_loss', 'eval_loss', 'learning_rate', 'grad_norm'])
    
    # Training
    global_step = 0
    best_eval_loss = float('inf')
    consecutive_nans = 0
    
    print("\n" + "="*80)
    print("STARTING TRAINING")
    print("="*80)
    
    for epoch in range(cfg.NUM_EPOCHS):
        print(f"\n{'='*80}\nEPOCH {epoch + 1}/{cfg.NUM_EPOCHS}\n{'='*80}")
        
        model.train()
        epoch_loss = 0.0
        optimizer.zero_grad()
        batches_processed = 0
        
        for batch_idx, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            
            loss, _ = model(input_ids, labels)
            
            # Check for NaN
            if torch.isnan(loss) or torch.isinf(loss):
                consecutive_nans += 1
                print(f"  ⚠️  NaN/Inf at step {global_step} (consecutive: {consecutive_nans})")
                if consecutive_nans >= cfg.MAX_CONSECUTIVE_NANS:
                    print(f"  ❌ Stopping due to instability")
                    csv_file.close()
                    return
                optimizer.zero_grad()
                continue
            
            consecutive_nans = 0
            loss = loss / cfg.GRADIENT_ACCUMULATION_STEPS
            loss.backward()
            
            epoch_loss += loss.item()
            batches_processed += 1
            
            if (batch_idx + 1) % cfg.GRADIENT_ACCUMULATION_STEPS == 0:
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.MAX_GRAD_NORM)
                
                lr = get_lr(global_step, cfg, total_steps, warmup_steps)
                for param_group in optimizer.param_groups:
                    param_group['lr'] = lr
                
                optimizer.step()
                optimizer.zero_grad()
                global_step += 1
                
                current_loss = loss.item() * cfg.GRADIENT_ACCUMULATION_STEPS
                
                # Log every step
                if global_step % 10 == 0:
                    print(f"  Step {global_step:4d} | Loss: {current_loss:.4f} | LR: {lr:.2e}")
                
                # Evaluate periodically
                if global_step % cfg.EVAL_STEPS == 0:
                    eval_loss = evaluate_loss(model, test_loader, device)
                    csv_writer.writerow([global_step, epoch + 1, current_loss, eval_loss, lr, grad_norm])
                    csv_file.flush()
                    
                    print(f"  → Eval loss: {eval_loss:.4f}")
                    
                    if eval_loss < best_eval_loss:
                        best_eval_loss = eval_loss
                        save_checkpoint(model, optimizer, epoch + 1, global_step, current_loss, eval_loss, None, cfg, "best_model.pt")
                        print(f"  → ✨ New best model saved!")
            
            if batch_idx % 100 == 0:
                torch.cuda.empty_cache()
        
        # End of epoch
        print(f"\n{'-'*80}\nEPOCH {epoch + 1} SUMMARY\n{'-'*80}")
        
        avg_train_loss = (epoch_loss * cfg.GRADIENT_ACCUMULATION_STEPS) / batches_processed
        eval_loss = evaluate_loss(model, test_loader, device)
        
        print(f"Train loss: {avg_train_loss:.4f}")
        print(f"Eval loss:  {eval_loss:.4f}")
        
        # Compute ROUGE per epoch
        rouge_scores = compute_rouge(model, test_dataset, sp, cfg, device)
        print(f"\nROUGE Scores (on {cfg.ROUGE_SAMPLE_SIZE} samples):")
        print(f"  ROUGE-1: {rouge_scores['rouge1']:.2f}")
        print(f"  ROUGE-2: {rouge_scores['rouge2']:.2f}")
        print(f"  ROUGE-L: {rouge_scores['rougeL']:.2f}")
        
        # Save epoch checkpoint
        path = save_checkpoint(model, optimizer, epoch + 1, global_step, avg_train_loss, eval_loss, rouge_scores, cfg, f"epoch_{epoch + 1}.pt")
        print(f"✓ Checkpoint: {path}")
    
    csv_file.close()
    print(f"\n✓ Training complete! Metrics saved to {csv_path}")
    print(f"✓ Best eval loss: {best_eval_loss:.4f}")

if __name__ == "__main__":
    train()

# GPT-2 Fine-tuning for Nepali Summarization - Improved Version


## 1. CONFIGURATION

In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from tqdm import tqdm
import sentencepiece as spm
from datetime import datetime
import math

class Config:
    # Paths
    CHECKPOINT_PATH = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\new-models\bpe16_updated_19k.pt"
    TOKENIZER_PATH = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\new-tokenizers\bpe-16-updated.model"
    TRAIN_JSONL = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\train_plus_val_norm.jsonl"  # Updated to normalized dataset
    TEST_JSONL = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\test_norm.jsonl"  # Updated to normalized dataset
    OUTPUT_DIR = "finetuned-summarization"
    
    # Model settings
    BLOCK_SIZE = 1024  # Model hard limit
    MAX_SOURCE_LENGTH = 768  # 75% of block size for article + prompt
    MAX_TARGET_LENGTH = 256  # 25% of block size for summary
    
    # Freezing strategy: CORRECTED - No embedding freeze to avoid weight tying issues
    FREEZE_EMBEDDINGS = False  # Keeps both wte and lm_head trainable
    FREEZE_BLOCKS = list(range(8))  # Freeze first 8 blocks (0-7), train last 4 (8-11)
    
    # Training hyperparameters - IMPROVED
    NUM_EPOCHS = 5
    BATCH_SIZE = 4
    GRADIENT_ACCUMULATION_STEPS = 2
    LEARNING_RATE = 1e-5  # REDUCED from 3e-5 for stability
    WARMUP_RATIO = 0.1
    MAX_GRAD_NORM = 1.0
    WEIGHT_DECAY = 0.01
    
    # Generation settings - NEW
    GENERATION_TOP_K = 40
    GENERATION_TEMPERATURE = 0.7
    
    # Evaluation
    EVAL_STEPS = 100
    
    # Safety settings - NEW
    MAX_CONSECUTIVE_NANS = 3  # Stop if 3 consecutive NaN batches

config = Config()
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. MODEL ARCHITECTURE


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1
        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 16384
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, input_ids, labels=None):
        B, T = input_ids.size()
        assert T <= self.config.block_size, f"Sequence length {T} exceeds block size {self.config.block_size}"
        
        pos = torch.arange(0, T, dtype=torch.long, device=input_ids.device)
        pos_emb = self.transformer.wpe(pos)
        tok_emb = self.transformer.wte(input_ids)
        x = tok_emb + pos_emb
        
        for block in self.transformer.h:
            x = block(x)
        
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        
        return loss, logits

## 3. LOAD MODEL & APPLY FREEZING


In [4]:
print("\n" + "="*80)
print("LOADING MODEL")
print("="*80)

sp = spm.SentencePieceProcessor()
sp.load(config.TOKENIZER_PATH)
print(f"✓ Tokenizer loaded (vocab: {sp.vocab_size()})")

checkpoint = torch.load(config.CHECKPOINT_PATH, map_location=device, weights_only=False)
model_config = checkpoint['config']
model = GPT(model_config)
model.load_state_dict(checkpoint['model'])
model.to(device)
print(f"✓ Model loaded (step: {checkpoint['step']}, val_loss: {checkpoint['val_loss']:.4f})")

# Apply freezing strategy
print("\n" + "="*80)
print("APPLYING FREEZING STRATEGY")
print("="*80)

# Only freeze position embeddings (token embeddings remain trainable with lm_head)
if config.FREEZE_EMBEDDINGS:
    for param in model.transformer.wte.parameters():
        param.requires_grad = False
    for param in model.transformer.wpe.parameters():
        param.requires_grad = False
    print("✓ Frozen: Token & Position Embeddings")
else:
    # Only freeze position embeddings
    for param in model.transformer.wpe.parameters():
        param.requires_grad = False
    print("✓ Frozen: Position Embeddings only")
    print("✓ Token Embeddings (wte) remain trainable (shares weights with lm_head)")

# Freeze specified transformer blocks
for block_idx in config.FREEZE_BLOCKS:
    for param in model.transformer.h[block_idx].parameters():
        param.requires_grad = False
print(f"✓ Frozen: Blocks {config.FREEZE_BLOCKS[0]}-{config.FREEZE_BLOCKS[-1]}")
print(f"✓ Trainable: Blocks {config.FREEZE_BLOCKS[-1]+1}-{model_config.n_layer-1} + Final LayerNorm + LM Head")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\n✓ Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")


LOADING MODEL
✓ Tokenizer loaded (vocab: 16384)
✓ Model loaded (step: 19051, val_loss: 2.6268)

APPLYING FREEZING STRATEGY
✓ Frozen: Position Embeddings only
✓ Token Embeddings (wte) remain trainable (shares weights with lm_head)
✓ Frozen: Blocks 0-7
✓ Trainable: Blocks 8-11 + Final LayerNorm + LM Head

✓ Trainable parameters: 40,935,936 / 98,425,344 (41.6%)

LOADING DATASET
  Loaded 8080 samples from train_plus_val_norm.jsonl
  Loaded 858 samples from test_norm.jsonl

✓ Train samples: 8080
✓ Test samples:  858
✓ Train batches per epoch: 2020
✓ Optimizer steps per epoch: ~1010


## 4. DATASET PREPARATION


In [ ]:
print("\n" + "="*80)
print("LOADING DATASET")
print("="*80)

# UPDATED: Nepali prompt template
PROMPT_TEMPLATE = "यो लेखको संक्षेप गर्नुहोस्:\n{text}\nसारांश:\n"

class SummarizationDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, block_size):
        self.tokenizer = tokenizer
        self.block_size = block_size
        
        self.data = []
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    item = json.loads(line)
                    if 'text' in item and 'summary' in item:
                        self.data.append(item)
        
        print(f"  Loaded {len(self.data)} samples from {os.path.basename(jsonl_path)}")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Build full sequence (normalized dataset should already fit in block_size)
        prompt = PROMPT_TEMPLATE.format(text=item['text'])
        full_text = prompt + item['summary']
        
        # Tokenize
        tokens = self.tokenizer.encode(full_text)
        
        # Safety check: truncate if still exceeds (shouldn't happen with normalized data)
        if len(tokens) > self.block_size:
            print(f"  ⚠️  Sample {idx} exceeds block_size ({len(tokens)} > {self.block_size}), truncating")
            tokens = tokens[:self.block_size]
        
        input_ids = torch.tensor(tokens, dtype=torch.long)
        labels = input_ids.clone()
        
        # Mask prompt tokens in labels (only compute loss on summary)
        prompt_len = len(self.tokenizer.encode(prompt))
        labels[:prompt_len] = -100
        
        return {
            'input_ids': input_ids,
            'labels': labels,
            'text': item['text'],
            'summary': item['summary']
        }

train_dataset = SummarizationDataset(config.TRAIN_JSONL, sp, config.BLOCK_SIZE)
test_dataset = SummarizationDataset(config.TEST_JSONL, sp, config.BLOCK_SIZE)

print(f"\n✓ Train samples: {len(train_dataset)}")
print(f"✓ Test samples:  {len(test_dataset)}")

def collate_fn(batch):
    max_len = max(len(item['input_ids']) for item in batch)
    
    input_ids = []
    labels = []
    
    for item in batch:
        seq_len = len(item['input_ids'])
        pad_len = max_len - seq_len
        
        input_ids.append(torch.cat([item['input_ids'], torch.zeros(pad_len, dtype=torch.long)]))
        labels.append(torch.cat([item['labels'], torch.full((pad_len,), -100, dtype=torch.long)]))
    
    return {
        'input_ids': torch.stack(input_ids),
        'labels': torch.stack(labels)
    }

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"✓ Train batches per epoch: {len(train_loader)}")
print(f"✓ Optimizer steps per epoch: ~{len(train_loader) // config.GRADIENT_ACCUMULATION_STEPS}")

## 5. OPTIMIZER & SCHEDULER


In [5]:
print("\n" + "="*80)
print("OPTIMIZER SETUP")
print("="*80)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=config.LEARNING_RATE,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=config.WEIGHT_DECAY
)

total_steps = config.NUM_EPOCHS * len(train_loader) // config.GRADIENT_ACCUMULATION_STEPS
warmup_steps = int(config.WARMUP_RATIO * total_steps)

def get_lr(step):
    if step < warmup_steps:
        return config.LEARNING_RATE * (step + 1) / warmup_steps
    if step >= total_steps:
        return config.LEARNING_RATE * 0.1
    decay_ratio = (step - warmup_steps) / (total_steps - warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return config.LEARNING_RATE * 0.1 + coeff * (config.LEARNING_RATE * 0.9)

print(f"✓ Optimizer: AdamW")
print(f"✓ Learning rate: {config.LEARNING_RATE} (REDUCED for stability)")
print(f"✓ Weight decay: {config.WEIGHT_DECAY}")
print(f"✓ Total training steps: {total_steps}")
print(f"✓ Warmup steps: {warmup_steps}")
print(f"✓ Gradient accumulation: {config.GRADIENT_ACCUMULATION_STEPS}")
print(f"✓ Effective batch size: {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS}")


OPTIMIZER SETUP
✓ Optimizer: AdamW
✓ Learning rate: 1e-05 (REDUCED for stability)
✓ Weight decay: 0.01
✓ Total training steps: 5050
✓ Warmup steps: 505
✓ Gradient accumulation: 2
✓ Effective batch size: 8


## 6. EVALUATION FUNCTIONS


In [ ]:
def evaluate_loss(model, dataloader):
    """Compute average loss on evaluation set"""
    model.eval()
    total_loss = 0.0
    count = 0
    
    with torch.no_grad():
        for batch in dataloader:
            try:
                input_ids = batch['input_ids'].to(device)
                labels = batch['labels'].to(device)
                
                loss, _ = model(input_ids, labels)
                if not torch.isnan(loss) and not torch.isinf(loss):
                    total_loss += loss.item()
                    count += 1
            except RuntimeError as e:
                print(f"  ⚠️  Error during evaluation: {e}")
                continue
    
    model.train()
    return total_loss / count if count > 0 else float('nan')

def generate_summary_topk(model, tokenizer, text, max_new_tokens=128, top_k=40, temperature=0.7):
    """Generate summary using top-k sampling (IMPROVED)"""
    model.eval()
    prompt = PROMPT_TEMPLATE.format(text=text)
    tokens = tokenizer.encode(prompt)
    input_ids = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)
    
    generated = input_ids.clone()
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            if generated.size(1) >= model.config.block_size:
                break
            
            _, logits = model(generated)
            logits = logits[:, -1, :] / temperature
            
            # Top-k sampling
            top_k_logits, top_k_indices = torch.topk(logits, min(top_k, logits.size(-1)), dim=-1)
            probs = F.softmax(top_k_logits, dim=-1)
            next_token_idx = torch.multinomial(probs, 1)
            next_token = torch.gather(top_k_indices, -1, next_token_idx)
            
            generated = torch.cat([generated, next_token], dim=1)
            
            # Stop at EOS or padding token
            if next_token.item() == 0:
                break
    
    summary = tokenizer.decode(generated[0, len(tokens):].tolist())
    model.train()
    return summary

def generate_summary_greedy(model, tokenizer, text, max_new_tokens=64):
    """Generate summary using greedy decoding (fallback)"""
    model.eval()
    prompt = PROMPT_TEMPLATE.format(text=text)
    tokens = tokenizer.encode(prompt)
    input_ids = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)
    
    generated = input_ids.clone()
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            if generated.size(1) >= model.config.block_size:
                break
            
            _, logits = model(generated)
            next_token = torch.argmax(logits[:, -1, :], dim=-1)
            generated = torch.cat([generated, next_token.unsqueeze(0)], dim=1)
            
            if next_token.item() == 0:
                break
    
    summary = tokenizer.decode(generated[0, len(tokens):].tolist())
    model.train()
    return summary

def compute_rouge(model, dataset, sample_size=None, use_topk=True):
    """Compute ROUGE scores on dataset"""
    try:
        from rouge_score import rouge_scorer
    except ImportError:
        print("⚠️  rouge_score not installed. Run: pip install rouge-score")
        return None
    
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    
    # Use sample if dataset is large
    eval_size = sample_size if sample_size and sample_size < len(dataset) else len(dataset)
    
    print(f"  Computing ROUGE on {eval_size} samples (method: {'top-k' if use_topk else 'greedy'})...")
    for i in tqdm(range(eval_size), desc="ROUGE"):
        item = dataset.data[i]
        
        if use_topk:
            pred = generate_summary_topk(
                model, sp, item['text'], 
                config.MAX_TARGET_LENGTH,
                config.GENERATION_TOP_K,
                config.GENERATION_TEMPERATURE
            )
        else:
            pred = generate_summary_greedy(model, sp, item['text'], config.MAX_TARGET_LENGTH)
        
        ref = item['summary']
        
        result = scorer.score(ref, pred)
        scores['rouge1'].append(result['rouge1'].fmeasure)
        scores['rouge2'].append(result['rouge2'].fmeasure)
        scores['rougeL'].append(result['rougeL'].fmeasure)
    
    return {k: sum(v) / len(v) * 100 for k, v in scores.items()}

## 7. TRAINING LOOP


In [ ]:
print("\n" + "="*80)
print("STARTING TRAINING")
print("="*80)

global_step = 0
best_eval_loss = float('inf')
history = []
consecutive_nans = 0  # Track consecutive NaN batches

for epoch in range(config.NUM_EPOCHS):
    print(f"\n{'='*80}")
    print(f"EPOCH {epoch + 1}/{config.NUM_EPOCHS}")
    print(f"{'='*80}")
    
    model.train()
    epoch_loss = 0.0
    optimizer.zero_grad()
    batches_processed = 0
    
    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward pass
        loss, _ = model(input_ids, labels)
        
        # Check for NaN or Inf
        if torch.isnan(loss) or torch.isinf(loss):
            consecutive_nans += 1
            print(f"  ⚠️  {'NaN' if torch.isnan(loss) else 'Inf'} loss detected at step {global_step} (consecutive: {consecutive_nans})")
            
            if consecutive_nans >= config.MAX_CONSECUTIVE_NANS:
                print(f"  ❌ Too many consecutive NaN/Inf losses ({consecutive_nans}). Stopping training.")
                print(f"  💡 Suggestions:")
                print(f"     - Check if normalized dataset has issues")
                print(f"     - Further reduce learning rate")
                print(f"     - Check for data corruption")
                break
            
            optimizer.zero_grad()
            continue
        
        # Reset NaN counter on successful batch
        consecutive_nans = 0
        
        # Backward pass with gradient accumulation
        loss = loss / config.GRADIENT_ACCUMULATION_STEPS
        loss.backward()
        
        epoch_loss += loss.item()
        batches_processed += 1
        
        # Optimizer step after accumulation
        if (batch_idx + 1) % config.GRADIENT_ACCUMULATION_STEPS == 0:
            # Gradient clipping
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), config.MAX_GRAD_NORM)
            
            # Check for exploding gradients
            if grad_norm > config.MAX_GRAD_NORM * 10:
                print(f"  ⚠️  Large gradient norm detected: {grad_norm:.2f}")
            
            lr = get_lr(global_step)
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr
            
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1
            
            # Logging
            if global_step % 10 == 0:
                current_loss = loss.item() * config.GRADIENT_ACCUMULATION_STEPS
                print(f"  Step {global_step:4d} | Loss: {current_loss:.4f} | LR: {lr:.2e} | Grad: {grad_norm:.2f}")
            
            # Step-wise evaluation
            if global_step % config.EVAL_STEPS == 0:
                print(f"\n  {'─'*76}")
                print(f"  EVALUATION AT STEP {global_step}")
                print(f"  {'─'*76}")
                
                eval_loss = evaluate_loss(model, test_loader)
                print(f"  → Eval loss: {eval_loss:.4f}")
                
                if eval_loss < best_eval_loss:
                    best_eval_loss = eval_loss
                    print(f"  → ✨ New best eval loss!")
                    
                    # Save best model
                    best_model_path = os.path.join(config.OUTPUT_DIR, "best_model.pt")
                    torch.save({
                        'model': model.state_dict(),
                        'config': model.config,
                        'optimizer': optimizer.state_dict(),
                        'epoch': epoch + 1,
                        'global_step': global_step,
                        'eval_loss': eval_loss,
                    }, best_model_path)
                    print(f"  → Saved best model: {best_model_path}")
                
                print(f"  {'─'*76}\n")
        
        # Clear CUDA cache periodically
        if batch_idx % 100 == 0:
            torch.cuda.empty_cache()
    
    # Check if training was stopped due to NaNs
    if consecutive_nans >= config.MAX_CONSECUTIVE_NANS:
        print(f"\n❌ Training stopped early at epoch {epoch + 1} due to instability")
        break
    
    # End of epoch evaluation
    print(f"\n{'-'*80}")
    print(f"EPOCH {epoch + 1} SUMMARY")
    print(f"{'-'*80}")
    
    avg_train_loss = (epoch_loss * config.GRADIENT_ACCUMULATION_STEPS) / batches_processed if batches_processed > 0 else float('nan')
    eval_loss = evaluate_loss(model, test_loader)
    
    print(f"Average train loss: {avg_train_loss:.4f}")
    print(f"Evaluation loss:    {eval_loss:.4f}")
    
    # Compute ROUGE scores (use top-k sampling)
    rouge_scores = compute_rouge(model, test_dataset, sample_size=100, use_topk=True)
    if rouge_scores:
        print(f"\nROUGE Scores (on 100 test samples with top-k sampling):")
        print(f"  ROUGE-1: {rouge_scores['rouge1']:.2f}")
        print(f"  ROUGE-2: {rouge_scores['rouge2']:.2f}")
        print(f"  ROUGE-L: {rouge_scores['rougeL']:.2f}")
    
    # Save epoch checkpoint
    checkpoint_path = os.path.join(config.OUTPUT_DIR, f"epoch_{epoch + 1}.pt")
    torch.save({
        'model': model.state_dict(),
        'config': model.config,
        'optimizer': optimizer.state_dict(),
        'epoch': epoch + 1,
        'global_step': global_step,
        'train_loss': avg_train_loss,
        'eval_loss': eval_loss,
        'rouge_scores': rouge_scores,
    }, checkpoint_path)
    print(f"\n✓ Checkpoint saved: {checkpoint_path}")
    
    # Update history
    history.append({
        'epoch': epoch + 1,
        'global_step': global_step,
        'train_loss': avg_train_loss,
        'eval_loss': eval_loss,
        'rouge_scores': rouge_scores,
    })
    
    print(f"{'='*80}\n")


STARTING TRAINING

EPOCH 1/5
  Step   10 | Loss: 10.2085 | LR: 1.98e-07 | Grad: 8.00
  Step   20 | Loss: 9.3400 | LR: 3.96e-07 | Grad: 7.49
  Step   30 | Loss: 8.6089 | LR: 5.94e-07 | Grad: 6.87
  Step   40 | Loss: 9.5084 | LR: 7.92e-07 | Grad: 7.98
  Step   50 | Loss: 9.6538 | LR: 9.90e-07 | Grad: 7.60
  Step   60 | Loss: 9.6140 | LR: 1.19e-06 | Grad: 6.57
  Step   70 | Loss: 9.4636 | LR: 1.39e-06 | Grad: 6.66
  Step   80 | Loss: 9.4969 | LR: 1.58e-06 | Grad: 6.39
  Step   90 | Loss: 8.8848 | LR: 1.78e-06 | Grad: 5.05
  Step  100 | Loss: 9.1333 | LR: 1.98e-06 | Grad: 5.34

  ────────────────────────────────────────────────────────────────────────────
  EVALUATION AT STEP 100
  ────────────────────────────────────────────────────────────────────────────
  → Eval loss: 9.0901
  → ✨ New best eval loss!
  → Saved best model: finetuned-summarization\best_model.pt
  ────────────────────────────────────────────────────────────────────────────

  Step  110 | Loss: 9.1107 | LR: 2.18e-06 | Gra

ROUGE: 100%|██████████| 100/100 [06:26<00:00,  3.86s/it]



ROUGE Scores (on 100 test samples with top-k sampling):
  ROUGE-1: 0.00
  ROUGE-2: 0.00
  ROUGE-L: 0.00

✓ Checkpoint saved: finetuned-summarization\epoch_1.pt


EPOCH 2/5
  Step 1020 | Loss: 6.2433 | LR: 9.72e-06 | Grad: 4.43
  Step 1030 | Loss: 7.4202 | LR: 9.71e-06 | Grad: 4.62
  Step 1040 | Loss: 6.8840 | LR: 9.70e-06 | Grad: 4.87
  Step 1050 | Loss: 6.5528 | LR: 9.69e-06 | Grad: 5.40
  Step 1060 | Loss: 7.7051 | LR: 9.67e-06 | Grad: 4.45
  Step 1070 | Loss: 7.1782 | LR: 9.66e-06 | Grad: 5.00
  Step 1080 | Loss: 7.4826 | LR: 9.65e-06 | Grad: 5.34
  Step 1090 | Loss: 7.3139 | LR: 9.64e-06 | Grad: 4.27
  Step 1100 | Loss: 6.7566 | LR: 9.63e-06 | Grad: 4.77

  ────────────────────────────────────────────────────────────────────────────
  EVALUATION AT STEP 1100
  ────────────────────────────────────────────────────────────────────────────
  → Eval loss: 7.2891
  → ✨ New best eval loss!
  → Saved best model: finetuned-summarization\best_model.pt
  ─────────────────────────────────────

ROUGE: 100%|██████████| 100/100 [05:22<00:00,  3.22s/it]



ROUGE Scores (on 100 test samples with top-k sampling):
  ROUGE-1: 0.00
  ROUGE-2: 0.00
  ROUGE-L: 0.00

✓ Checkpoint saved: finetuned-summarization\epoch_2.pt


EPOCH 3/5
  Step 2030 | Loss: 6.5709 | LR: 7.73e-06 | Grad: 6.19
  Step 2040 | Loss: 6.6975 | LR: 7.70e-06 | Grad: 6.62
  Step 2050 | Loss: 5.8799 | LR: 7.67e-06 | Grad: 5.81
  Step 2060 | Loss: 7.0815 | LR: 7.64e-06 | Grad: 5.44
  Step 2070 | Loss: 6.3135 | LR: 7.62e-06 | Grad: 5.60
  Step 2080 | Loss: 7.7092 | LR: 7.59e-06 | Grad: 6.57
  Step 2090 | Loss: 7.1015 | LR: 7.56e-06 | Grad: 5.42
  Step 2100 | Loss: 7.3328 | LR: 7.53e-06 | Grad: 6.57

  ────────────────────────────────────────────────────────────────────────────
  EVALUATION AT STEP 2100
  ────────────────────────────────────────────────────────────────────────────
  → Eval loss: 6.9707
  → ✨ New best eval loss!
  → Saved best model: finetuned-summarization\best_model.pt
  ────────────────────────────────────────────────────────────────────────────

  Step 2110 | 

ROUGE: 100%|██████████| 100/100 [06:28<00:00,  3.89s/it]



ROUGE Scores (on 100 test samples with top-k sampling):
  ROUGE-1: 0.00
  ROUGE-2: 0.00
  ROUGE-L: 0.00

✓ Checkpoint saved: finetuned-summarization\epoch_3.pt


EPOCH 4/5
  Step 3040 | Loss: 6.9236 | LR: 4.69e-06 | Grad: 6.24
  Step 3050 | Loss: 6.3095 | LR: 4.66e-06 | Grad: 6.26
  Step 3060 | Loss: 6.6241 | LR: 4.63e-06 | Grad: 6.88
  Step 3070 | Loss: 6.7678 | LR: 4.60e-06 | Grad: 6.13
  Step 3080 | Loss: 7.0467 | LR: 4.57e-06 | Grad: 6.18
  Step 3090 | Loss: 6.4194 | LR: 4.54e-06 | Grad: 6.51
  Step 3100 | Loss: 6.8041 | LR: 4.51e-06 | Grad: 6.31

  ────────────────────────────────────────────────────────────────────────────
  EVALUATION AT STEP 3100
  ────────────────────────────────────────────────────────────────────────────
  → Eval loss: 6.8512
  ────────────────────────────────────────────────────────────────────────────

  Step 3110 | Loss: 6.1160 | LR: 4.48e-06 | Grad: 6.24
  Step 3120 | Loss: 6.2642 | LR: 4.45e-06 | Grad: 6.60
  Step 3130 | Loss: 7.0771 | LR: 4.42e-06 | G

ROUGE: 100%|██████████| 100/100 [05:27<00:00,  3.27s/it]



ROUGE Scores (on 100 test samples with top-k sampling):
  ROUGE-1: 0.00
  ROUGE-2: 0.00
  ROUGE-L: 0.00

✓ Checkpoint saved: finetuned-summarization\epoch_4.pt


EPOCH 5/5
  Step 4050 | Loss: 6.7908 | LR: 2.03e-06 | Grad: 6.63
  Step 4060 | Loss: 6.4368 | LR: 2.02e-06 | Grad: 7.07
  Step 4070 | Loss: 5.8405 | LR: 2.00e-06 | Grad: 7.45
  Step 4080 | Loss: 6.9629 | LR: 1.98e-06 | Grad: 6.33
  Step 4090 | Loss: 6.6235 | LR: 1.96e-06 | Grad: 7.94
  Step 4100 | Loss: 6.3476 | LR: 1.94e-06 | Grad: 6.81

  ────────────────────────────────────────────────────────────────────────────
  EVALUATION AT STEP 4100
  ────────────────────────────────────────────────────────────────────────────
  → Eval loss: 6.8039
  ────────────────────────────────────────────────────────────────────────────

  Step 4110 | Loss: 6.4435 | LR: 1.92e-06 | Grad: 6.03
  Step 4120 | Loss: 6.0069 | LR: 1.90e-06 | Grad: 7.14
  Step 4130 | Loss: 7.1535 | LR: 1.88e-06 | Grad: 6.76
  Step 4140 | Loss: 6.7661 | LR: 1.86e-06 | G

## 8. FINAL EVALUATION & SAVE RESULTS


In [ ]:
print("\n" + "="*80)
print("FINAL EVALUATION ON FULL TEST SET")
print("="*80)

final_rouge = compute_rouge(model, test_dataset, use_topk=True)  # Full test set with top-k
if final_rouge:
    print(f"\nFinal ROUGE Scores (with top-k sampling):")
    print(f"  ROUGE-1: {final_rouge['rouge1']:.2f}")
    print(f"  ROUGE-2: {final_rouge['rouge2']:.2f}")
    print(f"  ROUGE-L: {final_rouge['rougeL']:.2f}")

# Save training log
log_path = os.path.join(config.OUTPUT_DIR, "training_log.json")
with open(log_path, 'w', encoding='utf-8') as f:
    json.dump({
        'timestamp': datetime.now().isoformat(),
        'configuration': {
            'freeze_embeddings': config.FREEZE_EMBEDDINGS,
            'freeze_blocks': config.FREEZE_BLOCKS,
            'trainable_params': trainable,
            'total_params': total,
            'trainable_percentage': 100 * trainable / total,
            'num_epochs': config.NUM_EPOCHS,
            'batch_size': config.BATCH_SIZE,
            'gradient_accumulation': config.GRADIENT_ACCUMULATION_STEPS,
            'learning_rate': config.LEARNING_RATE,
            'warmup_ratio': config.WARMUP_RATIO,
            'generation_top_k': config.GENERATION_TOP_K,
            'generation_temperature': config.GENERATION_TEMPERATURE,
        },
        'training_history': history,
        'final_rouge_scores': final_rouge,
        'best_eval_loss': best_eval_loss,
    }, f, indent=2, ensure_ascii=False)

print(f"\n✓ Training log saved: {log_path}")

# ROUGE Score

In [ ]:
from rouge_score import rouge_scorer
"""
The cat sat on the mat in the morning.
आज बिहान बिरालो चटाईमा बसेको थियो।
"""
# Sample reference and generated summaries
reference_summary = "आज बिहान बिरालो चटाईमा बसेको थियो।"
generated_summary = "आज बिहान बिरालो चटाईमा बसेको थियो।"

# reference_summary = "बिरालो चटाईमा बसेको छ।"
# generated_summary = "बिरालो चटाईमा बसेको छ र सुतेको छ।"

# reference_summary = "The cat sat on the mat."
# generated_summary = "The cat is on the mat."

# Initialize ROUGE scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False) # Toggle

# Compute scores
scores = scorer.score(reference_summary, generated_summary)

# Print results
print("ROUGE-1:", scores['rouge1'])
print("ROUGE-2:", scores['rouge2'])
print("ROUGE-L:", scores['rougeL'])

ROUGE-1: Score(precision=0.0, recall=0.0, fmeasure=0.0)
ROUGE-2: Score(precision=0.0, recall=0.0, fmeasure=0.0)
ROUGE-L: Score(precision=0, recall=0, fmeasure=0)


In [11]:
from rouge_score import rouge_scorer
from rouge_score.tokenizers import Tokenizer
import re

class NepaliTokenizer(Tokenizer):
    def tokenize(self, text):
        text = text.replace("।", "")
        text = re.sub(r"[^\w\s]", "", text)
        return text.split()

tokenizer = NepaliTokenizer()

scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    tokenizer=tokenizer,
    use_stemmer=False
)
# reference_summary = "विद्युत चोरीमा संलग्न भएको आरोपमा विद्युत प्राधिकरणका कर्मचारी र व्यापारी गरी १६ जनलाई पक्रेर अनुसन्धान थालेको नेपाल प्रहरीले कारबाही प्रक्रियालाई अझ तीव्र बनाउने बताएको छ।"
# generated_summary = "नेपाल विद्युत प्राधिकरणले चोरीमा संलग्न रहेको भनिएका केही व्यापारीहरु अहिले पक्राउ परेपछि पक्राउ परेपनि चोरीको धन्दामा संलग्न भएको हुनसक्ने भन्दै प्रहरीले अनुसन्धान थाल्न थालेको छ। पक्राउ परेकाहरुमा केही व्यवसायी तथा केही अन्य व्यापारी रहेका छन्। उनीहरुमाथि आवश्यक अनुसन्धान सुरु गरेको प्राधिकरणले जनाएको छ। नेपाल विद्युत प्राधिकरणका अनुसार नेपालमा हाल उपलब्ध कुल विद्युतको एक चौथाइभन्दा बढी विद्युत चुहावट हुने गर्दछ। त्यसमा १२ प्रतिशत प्राविधिक तथा १४ प्रतिशत भन्दा बढी अप्राविधिक हुने गरेको छ। कमसल खालको विद्युतीय सामाग्री गर्दा हुने चुहावट प्राविधिक हो। मिटरमा कम खपत देखाउने गरी विद्युत चोरी भए त्यो चाहिँ अप्राविधिक चुहावटमा पर्छ। प्रयास चोरी नियन्त्रण गर्न उर्जा मन्त्रालयले छुट्टै समिति पनि गठन गरिएको छ। चोरी नियन्त्रणको अहिले थालिएको अभियानमा नेपाल विद्युत प्राधिकरण र उर्जा मन्त्रालयले सघाएको पनि प्रहरीले जनाएको छ। पक्राउ परेकाहरुलाई ठगी मुद्दा चलाइएको छ। तर उनीहरुलाई विद्युत चोरी न जस्ता आवश्यक न अन्तर्गत कारबाही अगाडि बढाउन सक्ने महाशाखा प्रमुख तथा एसएसपी खनाल बताउँछन्। उनले भने यसमा धेरै पक्षको संलग्नता भएकोले एकैथरी कानुनबाट सम्बोधन नहुन सक्छ। तर सबैजना ठगीसँग सम्बन्धित हुने भएकोले यो कानुनले समेट्छ। त्यही अनुरुप नै हामीले अनुसन्धान शुरु गरेका छौं। विद्युत चोरी गर्नेमा सर्वसाधारण उद्योगीहरु र व्यापारीहरु रहेको बताइएको छ। विद्युतको चोरी र चुहावट रोक्ने भनिदैं आएपनि हालसम्म त्यो प्रभावकारी देखिएको छैन। सारांश ⁇"

reference_summary = "आज बिहान बिरालो चटाईमा बसेको थियो विद्युत।"
generated_summary = "आज बिहान बिरालो चटाईमा बसेको थियो चोरीमा आरोपमा रहेको।"

scores = scorer.score(reference_summary, generated_summary)

print("ROUGE-1:", scores['rouge1'])
print("ROUGE-2:", scores['rouge2'])
print("ROUGE-L:", scores['rougeL'])

# ROUGE Score is F1 Score (fmeasure)

ROUGE-1: Score(precision=0.6666666666666666, recall=0.8571428571428571, fmeasure=0.75)
ROUGE-2: Score(precision=0.625, recall=0.8333333333333334, fmeasure=0.7142857142857143)
ROUGE-L: Score(precision=0.6666666666666666, recall=0.8571428571428571, fmeasure=0.75)
